#SCRAP, PRE PROCESS DATA

In [ ]:
!pip install google-play-scraper pandas tqdm langdetect deep-translator transformers torch emoji contractions gensim Sastrawi

In [ ]:
from google_play_scraper import reviews, Sort
import pandas as pd
from tqdm import tqdm
import datetime

app_id = "com.bibit.bibitid"
all_reviews = []
cursor = None

print("🚀 Scraping all possible reviews...")

while True:
    result, cursor = reviews(
        app_id,
        lang="id",
        country="id",
        sort=Sort.MOST_RELEVANT,
        count=200,
        continuation_token=cursor
    )

    if not result:
        break

    all_reviews.extend(result)

    if cursor is None:
        break

    if len(all_reviews) >= 50000:  # buffer besar
        break

print(f"Total scraped: {len(all_reviews)}")

filter time stamp, jumlah kata

In [ ]:
import pandas as pd

# ====== 1. DATA MENTAH ======
df_raw = pd.DataFrame(all_reviews)
df_raw["at"] = pd.to_datetime(df_raw["at"])

# ====== 2. FILTER TIMESTAMP (JAN 2022 – DES 2025) ======
start_date = pd.Timestamp("2021-01-01")
end_date   = pd.Timestamp("2025-01-31")

df_time = df_raw[
    (df_raw["at"] >= start_date) &
    (df_raw["at"] <= end_date)
].copy()

print("After time filter:", len(df_time))

# ====== 3. HITUNG JUMLAH KATA ======
df_time["wc_raw"] = (
    df_time["content"]
    .astype(str)
    .str.split()
    .str.len()
)

# ====== 4. FILTER MIN KATA ======
MIN_WORDS = 5

df_final = df_time[df_time["wc_raw"] >= MIN_WORDS].copy()

print(f"After min {MIN_WORDS} words filter:", len(df_final))

df_final.head(5)

In [ ]:
df_final.to_csv(
    "bibit_reviews_indonesian.csv",
    index=False,
    encoding="utf-8"
)

preproses

mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#case folding

import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/skripsi/bibit_reviews_indonesian_new.csv")
df['case_folding'] = df['content'].str.lower()

In [ ]:
#cleaning data
import re

def clean_text(text):
    text = str(text)

    # hapus URL
    text = re.sub(r'http\S+|www\S+', '', text)

    # hapus HTML
    text = re.sub(r'<.*?>', '', text)

    # hapus mention & hashtag
    text = re.sub(r'@\w+|#\w+', '', text)

    # hapus emoji
    text = re.sub(r'[^\x00-\x7F]+', '', text)

    # hapus tanda baca (termasuk .... !!! ,,,)
    text = re.sub(r'[^\w\s]', '', text)

    # hapus karakter aneh
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    return text

# terapkan cleaning
df['cleaning'] = df['case_folding'].apply(clean_text)

print("Cleaning selesai:", len(df))

In [ ]:
#normalisasi
# load kamus
kamus = {}
with open("/content/drive/MyDrive/skripsi/kamus_normalisasi.txt", "r", encoding="utf-8") as f:
    for line in f:
        kata = line.strip().split(",")
        if len(kata) == 2:
            kamus[kata[0].strip()] = kata[1].strip()

# fungsi normalisasi
def normalize_text(text):
    # pisahkan kata & tanda baca
    tokens = re.findall(r'\w+|[^\w\s]', text)

    # normalisasi kata
    tokens = [kamus[token] if token in kamus else token for token in tokens]

    # gabungkan kembali
    text = " ".join(tokens)

    # rapikan spasi sebelum tanda baca
    text = re.sub(r'\s+([?.!,])', r'\1', text)

    return text

df['cleaning'] = df['cleaning'].apply(normalize_text)

# terapkan
df['normalisasi'] = df['cleaning'].apply(normalize_text)
print("Normalisasi selesai")

df.to_csv('/content/drive/MyDrive/skripsi/bibit_reviews_normal.csv', index=False)

In [ ]:
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# stopword bawaan
factory = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

# custom stopword (single word)
custom_stopwords = {
    "yang", "dan", "di", "ke", "dari",
    "aja", "nih", "sih", "dong", "kok", "lah",
    "nya", "deh", "tuh", "kan", "tidak",
    "aku", "saya", "kamu", "dia", "mereka",
    "admin", "kak", "kakak", "sekali",
    "udah", "sudah", "belum", "jadi", "tibatiba",
    "banget", "doang", "saja", "kadangkadang",
    "dll", "dsb", "mudahmudahan", "padahal",
    "rt", "via", "mudahmudah", "garagara"
}

# multi-word stopwords
multi_stopwords = {
    "dan lain-lain",
    "dan lain sebagainya",
    "tiba tiba",
    "kadang kadang",
    "mudah mudahan",
    "tibatiba tibatiba",
    "tiba-tiba",
    "kadang-kadang",
    "mudah mudah",
    "mudah-mudahan",
    "mudah-mudah an",
    "mudah-mudah",
    "mudah mudahmudahan",
    "dana dana",
    "gara gara",
    "gara-gara"
}

# Suffix untuk dihapus
suffixes_to_remove = ["nya", "kan", "ku", "mu", "kah", "pun"]

# Kata dasar minimal 3 huruf
MIN_WORD_LENGTH = 3

# Gabungkan stopwords
all_stopwords = stopwords.union(custom_stopwords)

def remove_suffixes_from_word(word):
    """Hapus suffix dari kata (hanya sekali)"""
    word_lower = word.lower()

    for suffix in suffixes_to_remove:
        if word_lower.endswith(suffix):
            stripped_length = len(word_lower) - len(suffix)
            if stripped_length >= MIN_WORD_LENGTH:
                # Cek khusus: jangan hapus "i" dari "kami", "sini", "dari"
                if suffix == "i" and stripped_length == 3:
                    # Kata seperti "dari" (4 huruf) - suffix "i" -> "dar" (3 huruf)
                    # Biarkan saja jika memang cocok
                    pass
                return word_lower[:-len(suffix)]

    return word_lower

def remove_stopwords(text):
    # Handle berbagai tipe input
    if pd.isna(text):
        return ""
    if not isinstance(text, str):
        text = str(text)

    # Bersihkan karakter aneh
    text = text.strip()
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = text.replace('\t', ' ')

    # Step 1: Lowercase
    text = text.lower()

    # Step 2: Hapus tanda baca (tapi pertahankan spasi)
    text = re.sub(r'[^\w\s-]', ' ', text)

    # Step 3: Hapus multi-word stopwords
    for phrase in multi_stopwords:
        # Pattern dengan variasi spasi
        pattern = re.escape(phrase).replace(r'\ ', r'[\s-]+')
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE)

    # Step 4: Bersihkan spasi
    text = re.sub(r'\s+', ' ', text).strip()

    # Step 5: Proses per kata
    words = text.split()
    filtered_words = []

    for word in words:
        if not word or len(word) < 2:
            continue

        # Hapus suffix
        word_stripped = remove_suffixes_from_word(word)

        # Cek stopword
        if word_stripped in all_stopwords:
            continue

        # Kasus khusus "gara"
        if word_stripped == "gara":
            continue

        # Simpan hasil
        filtered_words.append(word_stripped if word_stripped != word else word)

    result = " ".join(filtered_words)
    result = re.sub(r'\s+', ' ', result).strip()

    return result

# Terapkan ke dataframe dengan progress bar
from tqdm import tqdm
tqdm.pandas()

print("Memproses stopword removal...")
df['stopword'] = df['normalisasi'].progress_apply(remove_stopwords)

# VERIFIKASI HASIL
print("\n=== VERIFIKASI HASIL ===")
print("Contoh perbandingan:")
for idx in range(min(10, len(df))):
    original = df.iloc[idx]['normalisasi']
    result = df.iloc[idx]['stopword']
    print(f"\n{idx+1}. Original: {original}")
    print(f"   Result:   {result}")
    print(f"   Sama? {'✓' if original != result else '✗ (tidak berubah)'}")

In [ ]:
# Test
if __name__ == "__main__":
    test_cases = [
        "aplikasinya bagus sekali",
        "penarikannya cepat",
        "gara-gara dia datang terlambat",
        "gara gara hujan deras",
        "bukunya saya baca",
        "jalannya macet parah",
        "makannya lahap banget",
        "dana dana ini untuk acara",
        "tiba-tiba dia pergi",
        "mudah-mudahan lulus",
        "keuntungannya setelah"
    ]

    print("=== TEST REMOVE STOPWORDS (FIXED) ===")
    for test in test_cases:
        result = remove_stopwords(test)
        print(f"Input : {test}")
        print(f"Output: {result}")
        print("-" * 50)

In [ ]:
#save csv
df.to_csv("/content/drive/MyDrive/skripsi/bibit_reviews_clean_stopword.csv",index=False)

print("Jumlah data akhir:", len(df))

In [ ]:
df

In [ ]:
#cek dan filter bahasa
from langdetect import detect

def detect_lang(text):
    try:
        return detect(text)
    except:
        return "unknown"

# deteksi bahasa
df['lang_id'] = df['normalisasi'].apply(detect_lang)

# jumlah sebelum
before_count = len(df)

# filter hanya Indonesia
df = df[df['lang_id'] == 'id']

# jumlah sesudah
after_count = len(df)

# tampilkan hasil
print("Jumlah sebelum filter bahasa:", before_count)
print("Jumlah setelah filter bahasa:", after_count)
print("Jumlah data terhapus:", before_count - after_count)

In [ ]:
#hapus duplikat
df = df.drop_duplicates(subset=['normalisasi'])

#hapus hanya simbol
df = df[df['normalisasi'].str.contains(r'[A-Za-z0-9]', regex=True)]

#hapus karakter berulang
df = df[~df['normalisasi'].str.contains(r'(.)\1{3,}', regex=True)]

#minimal kata
df = df[df['normalisasi'].str.split().str.len() >= 5]

print("Jumlah akhir setelah semua filtering:", len(df))

In [ ]:
import pandas as pd
import re

def clean_bibit_reviews_pipeline():

    # kata kunci spam
    spam_keywords = [
        # kode & referral
        'pakai kode', 'gunakan kode', 'kode referral', 'kode referal',
        'kode promo', 'kode undangan', 'kode saya', 'referral',
        'cashback', 'uang gratis', 'bonus', 'voucher', 'diskon',
        'hadiah', 'gratis saldo', 'saldo gratis', 'cash back',
        'daftar sekarang', 'gabung sekarang', 'ayo daftar',
        'klik di sini', 'klik link', 'join sekarang', 'gabung', 'join',
        'hubungi saya', 'chat saya', 'dm saya',
        'wa saya', 'whatsapp saya', 'kontak saya',
        'pasti untung', 'dijamin untung', 'tanpa risiko',
        'cepat kaya', 'cuan cepat',
        'giveaway', 'bagi bagi', 'bagi-bagi',
        'kode', 'kodeku', 'referralku'
    ]

    # fungsi deteksi spam
    def is_spam(text):
        text = str(text).lower()

        # kosong
        if text.strip() == '' or pd.isna(text):
            return True

        # keyword spam
        if any(k in text for k in spam_keywords):
            return True

        # mengandung URL/link
        if re.search(r'http\S+|www\S+', text):
            return True

        # terlalu banyak angka
        if sum(c.isdigit() for c in text) > 10:
            return True

        # terlalu sedikit kata bermakna
        if len([w for w in text.split() if len(w) > 2]) < 3:
            return True

        # karakter berulang (contoh: baguuuussssss)
        if re.search(r'(.)\1{3,}', text):
            return True

        # simbol/emoji berlebihan
        if len(re.findall(r'[^\w\s,]', text)) > 10:
            return True

        return False

    # deteksi spam
    df['is_spam'] = df['normalisasi'].apply(is_spam)

    # pisahkan data
    clean_df = df.loc[~df['is_spam']].copy()
    spam_df  = df.loc[df['is_spam']].copy()

    # output info
    print(f"Total reviews : {len(df)}")
    print(f"Clean reviews : {len(clean_df)}")
    print(f"Spam reviews  : {len(spam_df)}")

    return clean_df, spam_df


# jalankan
clean_df, spam_df = clean_bibit_reviews_pipeline()

In [ ]:
#save csv
clean_df.to_csv("/content/drive/MyDrive/skripsi/bibit_reviews_clean_final.csv",index=False)

print("Jumlah data akhir:", len(clean_df))

In [ ]:
clean_df

labeling sentimen

In [ ]:
import torch
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd

#load model pre trained
pretrained= "mdhugol/indonesia-bert-sentiment-classification"

#load model dan tokenizer
model = AutoModelForSequenceClassification.from_pretrained(pretrained)
tokenizer = AutoTokenizer.from_pretrained(pretrained)

#pipeline sentimen dan mapping sentimen
sentiment_analysis = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
label_index = {'LABEL_0': 'positive', 'LABEL_1': 'neutral', 'LABEL_2': 'negative'}

#load data
df = pd.read_csv("/content/drive/MyDrive/skripsi/bibit_reviews_clean_final.csv")

#fungsi sentimen
def indobert_sentiment(text):
    text = str(text) #pastikan teks memiliki tipe string

    result = sentiment_analysis(text, truncation=True)[0] #melakukan prediksi

    label = label_index[result['label']] #label sentimen
    score = result['score'] #skor sentimen

    return score, label

#menerapkan fungsi ke dataframe
df[["score", "sentiment"]] = df["normalisasi"].apply(
    lambda x: pd.Series(indobert_sentiment(x))
)

#simpan hasil
df.to_csv("/content/drive/MyDrive/skripsi/bibit_reviews_labelled.csv", index=False)

# cek hasil
print(df[["normalisasi", "score", "sentiment"]].head())

In [ ]:
df["sentiment"].value_counts()

In [ ]:
df["sentiment"].isna().sum()

In [ ]:
label_map = {"positive": 0, "neutral": 1, "negative": 2}
df["label"] = df["sentiment"].map(label_map)
df["label"].value_counts()

In [ ]:
# hitung proporsi persen
sentiment_percent = df["sentiment"].value_counts(normalize=True) * 100

print(sentiment_percent)

In [ ]:
import matplotlib.pyplot as plt

# Urutan label
order = ["positive", "neutral", "negative"]

# Hitung persentase
sentiment_percent = (
    df["sentiment"]
    .value_counts(normalize=True)
    .reindex(order) * 100
)

# Label tampilan
labels = ["Review Positif", "Review Netral", "Review Negatif"]

# Warna
colors = ["lightgreen", "lightskyblue", "lightcoral"]

# Figure
plt.figure(figsize=(5,5))

# Donut chart
wedges, texts, autotexts = plt.pie(
    sentiment_percent,
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    pctdistance=0.75,   # posisi persen di dalam lingkaran
    startangle=0,
    wedgeprops={
        'width': 0.40,
        'edgecolor': 'gray',
        'linewidth': 1.5
    },
    textprops={'fontsize': 9}
)

# Mempercantik teks persentase
for autotext in autotexts:
    autotext.set_color('black')
    autotext.set_fontsize(9)

# Judul
plt.title(
    "Proporsi Ulasan Aplikasi Bibit",
    fontsize=11
)

# Bentuk tetap bulat
plt.axis('equal')

plt.show()

split data

In [ ]:
import pandas as pd
df=pd.read_csv("/content/drive/MyDrive/skripsi/bibit_reviews_labelled.csv")

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

label_map = {"positive": 0, "neutral": 1, "negative": 2}
df["label"] = df["sentiment"].map(label_map)

df_train, df_temp = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)
df_val, df_test = train_test_split(
    df_temp,
    test_size=2/3,
    stratify=df_temp["label"],
    random_state=42
)
X_train = df_train["normalisasi"].values
y_train = df_train["label"].values

X_val = df_val["normalisasi"].values
y_val = df_val["label"].values

X_test = df_test["normalisasi"].values
y_test = df_test["label"].values

print("Train:", df_train.shape)
print("Val  :", df_val.shape)
print("Test :", df_test.shape)


In [ ]:
#cek distribusi data train, test, dan val
print("\nDistribusi Train:")
print(df_train["sentiment"].value_counts(normalize=True))

print("\nDistribusi Val:")
print(df_val["sentiment"].value_counts(normalize=True))

print("\nDistribusi Test:")
print(df_test["sentiment"].value_counts(normalize=True))

SMOTE

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
import pandas as pd

# TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vect = vectorizer.fit_transform(X_train).toarray()

# SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_vect, y_train)

# Cek distribusi
print("Distribusi label setelah SMOTE:")
print(pd.Series(y_train_bal).value_counts())

#IndoBERT

In [ ]:
!pip install transformers torch pandas scikit-learn tqdm seaborn

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import classification_report, confusion_matrix
from collections import defaultdict
import seaborn as sns
import matplotlib.pyplot as plt
import random
from sklearn.metrics import accuracy_score, f1_score

set seed

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data

In [ ]:
print("Distribusi data:")
print(df_train["label"].value_counts())

tokenizer

In [ ]:
model_name = "indobenchmark/indobert-base-p2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
MAX_LEN = 256

def tokenize_texts(texts):
    return tokenizer(
        list(texts),
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
        return_tensors='pt'
    )

In [ ]:
train_encodings = tokenize_texts(df_train["normalisasi"])
val_encodings   = tokenize_texts(df_val["normalisasi"])
test_encodings  = tokenize_texts(df_test["normalisasi"])

dataset class

In [ ]:
class BertDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
train_dataset = BertDataset(train_encodings, df_train["label"].values)
val_dataset   = BertDataset(val_encodings, df_val["label"].values)
test_dataset  = BertDataset(test_encodings, df_test["label"].values)

data loader

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

bert classifier

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

model.to(device)

In [ ]:
def tampilkan_proses_indobert(text, tokenizer, model, device):
    model.eval()

    # Tokenisasi
    encoding = tokenizer(
        text,
        return_tensors='pt',
        padding=False,
        truncation=True,
        return_token_type_ids=True
    )

    input_ids = encoding['input_ids'].to(device)

    # ambil token_type_ids (segment)
    token_type_ids = encoding.get('token_type_ids')
    if token_type_ids is not None:
        segment_ids = token_type_ids[0].cpu().tolist()
    else:
        segment_ids = None  # kalau model tidak pakai (misal RoBERTa)

    # Token (dengan CLS & SEP)
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu())

    # Hilangkan CLS & SEP untuk word tokenizing
    tokens_no_special = tokens[1:-1]

    # Token ID
    token_ids = input_ids[0].cpu().tolist()

    # Position (otomatis dari panjang token)
    position_ids = list(range(1, len(token_ids) + 1))

    # Output
    print("Original")
    print(text)

    print("\nToken Khusus")
    print("[CLS] + " + " ".join(tokens_no_special) + " + [SEP]")

    print("\nWord Tokenizing")
    print(tokens_no_special)

    print("\nToken Embedding")
    print(token_ids)

    print("\nSegmen Embedding")
    print(segment_ids)

    print("\nPosition Embedding")
    print(position_ids)

In [ ]:
sample_text = df_train["normalisasi"].iloc[0]

tampilkan_proses_indobert(sample_text, tokenizer, model, device)

optimizer, loss, & scheduler

In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-6)
EPOCHS = 5

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

training function

In [ ]:
def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()

    losses = []
    preds = []
    labels_all = []

    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        # backward
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        losses.append(loss.item())

        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())

    acc = accuracy_score(labels_all, preds)
    f1 = f1_score(labels_all, preds, average='weighted')
    loss_mean = np.mean(losses)

    return acc, f1, loss_mean

evaluation function

In [ ]:
def eval_model(model, data_loader, device):
    model.eval()

    losses = []
    preds = []
    labels_all = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            losses.append(loss.item())

            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    acc = accuracy_score(labels_all, preds)
    f1 = f1_score(labels_all, preds, average='weighted')
    loss_mean = np.mean(losses)

    return acc, f1, loss_mean

training loop

In [ ]:
history = defaultdict(list)
best_f1 = 0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 50)

    train_acc, train_f1, train_loss = train_epoch(
        model, train_loader, optimizer, scheduler, device
    )

    val_acc, val_f1, val_loss = eval_model(
        model, val_loader, device
    )

    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")

    # simpan history
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['train_loss'].append(train_loss)

    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['val_loss'].append(val_loss)

    # simpan best model berdasarkan F1 validation
    if val_f1 > best_f1:
        torch.save(model.state_dict(), "best_model.bin")
        best_f1 = val_f1
        print("Model terbaik disimpan")

In [ ]:
model.load_state_dict(torch.load("best_model.bin"))

test_acc, test_f1, test_loss = eval_model(model, test_loader, device)

print("\nHasil Test")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")

model evaluation

In [ ]:
plt.figure(figsize=(12,5))

# subplot 1 → Accuracy
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.title('Accuracy History')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.legend()

# subplot 2 → Loss
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Loss History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
def get_predictions(model, data_loader, device):
    model.eval()

    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    return predictions, true_labels

classification report

In [ ]:
y_pred, y_true = get_predictions(model, test_loader, device)

print("\nClassification Report Testing Data")
print(classification_report(y_true, y_pred, target_names=["positif","netral","negatif"]))

In [ ]:
y_pred_train, y_train_true = get_predictions(model, train_loader, device)

print("\nClassification Report Training Data")
print(classification_report(y_train_true,y_pred_train,target_names=["positif", "netral", "negatif"]))

In [ ]:
y_pred_val, y_val_true = get_predictions(model, val_loader, device)

print("\nClassification Report Validation Data")
print(classification_report(y_val_true,y_pred_val,target_names=["positif", "netral", "negatif"]))

confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

class_names = ["Positif", "Netral", "Negatif"]
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Sentiment")
plt.ylabel("True Sentiment")
plt.title("Confusion Matrix")
plt.show()

Prediksi Ulasan Baru

In [ ]:
def predict_batch_with_score(texts, model, tokenizer, device, batch_size=16):
    model.eval()

    all_preds = []
    all_scores = []

    label_map = {0: "positif", 1: "netral", 2: "negatif"}

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        encoding = tokenizer(
            batch,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=256
        )

        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        for j in range(len(preds)):
            label = label_map[preds[j].item()]
            score = probs[j][preds[j]].item()

            all_preds.append(label)
            all_scores.append(score)

    return all_preds, all_scores

In [ ]:
ulasan_baru = [
    "kecewa sama aplikasi nya, mau masuk dr pagi dibuka sampai sore gak bisa masuk, selalu jaringan tidak stabil, sedangkan aplikasi laen semua bisa dibuka gak ada masalah apapun",
    "sangat baik dan membantu. ulasan tambahan adalah inilah aplikasi yg membuat tidak ada kata terlambat untuk belajar,jadi seperti gen z masa kini walau masih tertatih-tatih",
    "sudah bagus dan mantap harapan nya semoga kedepan nya bisa ada fitur bibit international jadi ada pembelian saham luar negeri",
    "Awalnya nyaman2 aja pake bibit, namun ketika saya butuh menghubungi CS responnya lambat sekali, mau ngobrol live chat gak di layanin. Tolong untuk layanan CS di perbaiki.",
    "sebelumnya ga pernah nyalain fitur SIP tiba-tiba saya punya saldo terakhir hilang masuk ke bibit saya, katanya fitur SIP dan giliran mau saya tarik lagi gabisa dan itupun gabisa dibatalin juga"
]

In [ ]:
preds, scores = predict_batch_with_score(
    ulasan_baru, model, tokenizer, device
)

df_hasil = pd.DataFrame({
    "ulasan": ulasan_baru,
    "sentimen": preds,
    "score": scores
})

print(df_hasil)

#XLNet

In [ ]:
!pip install transformers torch pandas scikit-learn tqdm seaborn

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from torch.utils.data import TensorDataset, DataLoader
from transformers import XLNetTokenizer, XLNetForSequenceClassification
from torch.optim import AdamW
from tqdm import trange

device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

data

In [ ]:
print("Distribusi data:")
print(df_train["label"].value_counts())

tokenisasi

In [ ]:
model_name = "xlnet-base-cased"
tokenizer = XLNetTokenizer.from_pretrained(model_name)
MAX_LEN = 256

# Split data train_balanced menjadi train dan validation
X_train, X_val, y_train, y_val = train_test_split(
    df_train["normalisasi"],
    df_train["label"],
    test_size=0.1,
    random_state=42,
    stratify=df_train["label"]
)

# fungsi tokenisasi
def tokenize_texts(texts, tokenizer, max_len=MAX_LEN):
    encoding = tokenizer(
        texts.tolist() if isinstance(texts, pd.Series) else texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors='pt'
    )
    return encoding['input_ids'], encoding['attention_mask']

# tokenisasi
train_inputs, train_masks = tokenize_texts(X_train, tokenizer)
val_inputs, val_masks     = tokenize_texts(X_val, tokenizer)
test_inputs, test_masks   = tokenize_texts(df_test["normalisasi"], tokenizer)

# label → tensor
train_labels = torch.tensor(y_train.values, dtype=torch.long)
val_labels   = torch.tensor(y_val.values, dtype=torch.long)
test_labels  = torch.tensor(df_test["label"].values, dtype=torch.long)

In [ ]:
# contoh tokenisasi
text = df_train.iloc[0]["normalisasi"]

tokens = tokenizer.tokenize(text)
inputs = tokenizer(text, padding="max_length", truncation=True, max_length=30)

print("Teks:", text)
print("Tokens:", tokens)
print("Input IDs:", inputs["input_ids"])
print("Attention Mask:", inputs["attention_mask"])

data loader

In [ ]:
train_dataset = TensorDataset(train_inputs, train_masks, train_labels)
val_dataset   = TensorDataset(val_inputs, val_masks, val_labels)
test_dataset  = TensorDataset(test_inputs, test_masks, test_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

model

In [ ]:
model = XLNetForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

model.to(device)

optimizer

In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 5

fungsi akurasi

In [ ]:
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1)
    return np.sum(pred_flat == labels) / len(labels)

training function

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    preds_all, labels_all = [], []

    for batch in data_loader:
        b_input_ids, b_input_mask, b_labels = [t.to(device) for t in batch]

        optimizer.zero_grad()

        outputs = model(
            input_ids=b_input_ids,
            attention_mask=b_input_mask,
            labels=b_labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        preds = logits.argmax(dim=1).detach().cpu().numpy()
        labels = b_labels.detach().cpu().numpy()

        preds_all.extend(preds)
        labels_all.extend(labels)

    acc = accuracy_score(labels_all, preds_all)
    f1 = f1_score(labels_all, preds_all, average='weighted')

    return acc, f1, total_loss / len(data_loader)

evaluation function

In [ ]:
def eval_model(model, data_loader, device):
    model.eval()
    total_loss = 0
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in data_loader:
            b_input_ids, b_input_mask, b_labels = [t.to(device) for t in batch]

            outputs = model(
                input_ids=b_input_ids,
                attention_mask=b_input_mask,
                labels=b_labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = logits.argmax(dim=1).detach().cpu().numpy()
            labels = b_labels.detach().cpu().numpy()

            preds_all.extend(preds)
            labels_all.extend(labels)

    acc = accuracy_score(labels_all, preds_all)
    f1 = f1_score(labels_all, preds_all, average='weighted')

    return acc, f1, total_loss / len(data_loader)

training loop

In [ ]:
from collections import defaultdict
import torch
from transformers import get_linear_schedule_with_warmup

history = defaultdict(list)
best_f1 = 0

# Define scheduler
total_steps_xlnet = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps_xlnet),
    num_training_steps=total_steps_xlnet
)

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    print("-" * 50)

    train_acc, train_f1, train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_acc, val_f1, val_loss = eval_model(model, val_loader, device)

    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")

    # simpan history
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['train_loss'].append(train_loss)

    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['val_loss'].append(val_loss)

    # simpan best model
    if val_f1 > best_f1:
        torch.save(model.state_dict(), "best_model_xlnet.bin")
        best_f1 = val_f1
        print("Model terbaik disimpan")

In [ ]:
model.load_state_dict(torch.load('best_model_xlnet.bin'))

test_loss, test_acc, test_f1 = eval_model(
    model, test_loader, device
)

print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")

validasi

In [ ]:
model.eval()
eval_accuracy = 0

for batch in val_loader:
    b_input_ids, b_input_mask, b_labels = [t.to(device) for t in batch]

    with torch.no_grad():
        outputs = model(
            input_ids=b_input_ids,
            attention_mask=b_input_mask
        )

    logits = outputs.logits.cpu().numpy()
    label_ids = b_labels.cpu().numpy()
    eval_accuracy += flat_accuracy(logits, label_ids)

print(f"Validation Accuracy: {eval_accuracy / len(val_loader):.4f}")

model evaluation

In [ ]:
plt.figure(figsize=(12,5))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.title('Accuracy History')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.legend()

# Loss
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Loss History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
def get_predictions(model, data_loader, device):
    model.eval()

    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch[0].to(device)
            attention_mask = batch[1].to(device)
            labels = batch[2].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    return predictions, true_labels

klasifikasi report

In [ ]:
predictions, true_labels = get_predictions(model, test_loader, device)

print("\nClassification Report Testing Data")
print(classification_report(true_labels,predictions,target_names=["positif", "netral", "negatif"]))

In [ ]:
train_pred, train_true = get_predictions(model, train_loader, device)

print("\nClassification Report Training Data")
print(classification_report(train_true,train_pred,target_names=["positif", "netral", "negatif"]))

In [ ]:
val_true, val_pred = get_predictions(model, val_loader, device)

print("\nClassification Report (Validation Data")
print(classification_report(val_true,val_pred,target_names=["positif", "netral", "negatif"]))

confusion matrix

In [ ]:
class_names = ["Positif", "Netral", "Negatif"]
cm_xlnet = confusion_matrix(true_labels, predictions)
df_cm = pd.DataFrame(cm_xlnet, index=class_names, columns=class_names)

plt.figure(figsize=(6,5))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
plt.ylabel("True sentiment")
plt.xlabel("Predicted sentiment")
plt.tight_layout()
plt.show()

Prediksi Data Baru

In [ ]:
def predict_xlnet(texts, model, tokenizer, device, max_len=256, batch_size=16):
    model.eval()

    all_preds = []
    all_scores = []

    label_map = {0: "positif", 1: "netral", 2: "negatif"}

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        encoding = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        for j in range(len(preds)):
            all_preds.append(label_map[preds[j].item()])
            all_scores.append(probs[j][preds[j]].item())

    return all_preds, all_scores

In [ ]:
ulasan_baru = [
    "kecewa sama aplikasi nya, mau masuk dr pagi dibuka sampai sore gak bisa masuk, selalu jaringan tidak stabil, sedangkan aplikasi laen semua bisa dibuka gak ada masalah apapun",
    "sangat baik dan membantu. ulasan tambahan adalah inilah aplikasi yg membuat tidak ada kata terlambat untuk belajar,jadi seperti gen z masa kini walau masih tertatih-tatih",
    "sudah bagus dan mantap harapan nya semoga kedepan nya bisa ada fitur bibit international jadi ada pembelian saham luar negeri",
    "Awalnya nyaman2 aja pake bibit, namun ketika saya butuh menghubungi CS responnya lambat sekali, mau ngobrol live chat gak di layanin. Tolong untuk layanan CS di perbaiki.",
    "sebelumnya ga pernah nyalain fitur SIP tiba-tiba saya punya saldo terakhir hilang masuk ke bibit saya, katanya fitur SIP dan giliran mau saya tarik lagi gabisa dan itupun gabisa dibatalin juga"
]

In [ ]:
preds, scores = predict_xlnet(
    ulasan_baru, model, tokenizer, device
)

df_hasil = pd.DataFrame({
    "ulasan": ulasan_baru,
    "sentimen": preds,
    "score": scores
})

print(df_hasil)

In [ ]:
print(df_train.index[0])

#BERTopic

In [ ]:
!pip install bertopic sentence-transformers hdbscan umap-learn wordcloud Sastrawi gensim

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import pandas as pd
import numpy as np
import torch
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from transformers import AutoTokenizer, AutoModel

In [ ]:
import random
import numpy as np
import torch

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

load dataset

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/skripsi/bibit_reviews_labelled.csv")
documents = df["stopword"].astype(str).tolist()

In [ ]:
sample_text = documents[1173]

print("\nContoh Dokumen")
print(sample_text)

document embedding

In [ ]:
model_name = "indobenchmark/indobert-base-p2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
def tampilkan_embedding_indobert(text):
    encoding = tokenizer(text, return_tensors="pt", truncation=True).to(device)

    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0].cpu())
    token_ids = encoding["input_ids"][0].cpu().tolist()

    print("\n=== PROSES INDOBERT ===")
    print("Original:")
    print(text)

    print("\nToken Khusus:")
    print("[CLS] + " + " ".join(tokens[1:-1]) + " + [SEP]")

    print("\nWord Tokenizing:")
    print(tokens[1:-1])

    print("\nToken Embedding:")
    print(token_ids)

    print("\nPosition Embedding:")
    print(list(range(1, len(token_ids)+1)))

In [ ]:
tampilkan_embedding_indobert(sample_text)

In [ ]:
#fungsi embedding
def get_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    #mean pooling
    embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
    return embedding[0]

#embedding semua dokumen
embeddings = np.array([get_embedding(doc) for doc in documents])

In [ ]:
print("\nProses Embedding Semua Dokumen")

print("Shape embedding:", embeddings.shape)
print("Contoh embedding (5 dimensi pertama):")
print(embeddings[0][:5])

reduksi dimensi

In [ ]:
umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

clustering topik

In [ ]:
hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

tokenisasi & vektorisasi

In [ ]:
vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    max_df=0.95
)

model

In [ ]:
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    verbose=True
)

fit & transform bertopic

In [ ]:
topics, probabilities = topic_model.fit_transform(documents, embeddings)
topic_model.reduce_topics(documents, nr_topics=10)

informasi topik

In [ ]:
topic_info = topic_model.get_topic_info()
print(topic_info.head(10))

In [ ]:
topics_dict = topic_model.get_topics()
total_topics = len([t for t in topics_dict.keys() if t != -1])
print(f"\nTotal topik (tidak termasuk outlier): {total_topics}")

koherensi topik

In [ ]:
# Tokenisasi sederhana untuk koherensi (split kata)
tokenized_docs = [doc.split() for doc in documents]

# Buat dictionary untuk gensim
dictionary = Dictionary(tokenized_docs)

# Ambil topik (exclude -1/outlier)
topic_words = []
topic_ids = []

for topic_id in topics_dict:
    if topic_id != -1:
        words = [word for word, _ in topics_dict[topic_id]]
        topic_words.append(words)
        topic_ids.append(topic_id)

# Hitung coherence per topik
coherence_per_topic = []

for i, words in enumerate(topic_words):
    if len(words) > 1:  # Minimal 2 kata untuk coherence
        cm = CoherenceModel(
            topics=[words],
            texts=tokenized_docs,
            dictionary=dictionary,
            coherence='c_v'
        )
        score = cm.get_coherence()
        coherence_per_topic.append(score)
        print(f"Topik {topic_ids[i]}: {score:.4f}")
    else:
        print(f"Topik {topic_ids[i]}: SKIP (kurang dari 2 kata)")
        coherence_per_topic.append(0.0)

# Rata-rata coherence (hanya topik valid)
valid_scores = [s for s in coherence_per_topic if s > 0]
if valid_scores:
    avg_coherence = sum(valid_scores) / len(valid_scores)
    print(f"\n=== Rata-rata Coherence (c_v) ===")
    print(f"{avg_coherence:.4f}")

visualisasi

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_topics()     # overview
topic_model.visualize_barchart()  # topik dominan

wordcloud

In [ ]:
freq = {}

topics_data = topic_model.get_topics() # Get the dictionary of topics

for topic_id, word_scores_list in topics_data.items():
    if topic_id == -1: # Optionally, skip the outlier topic (-1) if it's not relevant for the overall word cloud
        continue
    for word, score in word_scores_list:
        if word in freq:
            freq[word] += score
        else:
            freq[word] = score

wc = WordCloud(
    width=1000,
    height=500,
    background_color="white",
    colormap="viridis"
).generate_from_frequencies(freq)

plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud BERTopic")
plt.show()

In [ ]:
topic_model.visualize_barchart(top_n_topics=10)

In [ ]:
# Informasi topik
topic_info = topic_model.get_topic_info()

# tampilkan tabel distribusi topik
topic_distribution_bt = topic_info[['Topic', 'Count', 'Name']]

topic_distribution_bt

In [ ]:
top_topics = topic_info[topic_info["Topic"] != -1].head(9)["Topic"].tolist()
for topic_id in top_topics:
    words = dict(topic_model.get_topic(int(topic_id)))
    if words:
        wc = WordCloud(width=800, height=400, background_color="white", colormap="viridis")
        wc.generate_from_frequencies(words)
        plt.figure(figsize=(8, 4))
        plt.imshow(wc, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"WordCloud Topik {topic_id}")
        plt.show()

dokumen per topik

In [ ]:
for topic_id in topic_info[topic_info["Topic"] != -1].head(15)["Topic"]:
    print(f"\nTOPIK {int(topic_id)}:")
    print("Kata kunci:", topic_model.get_topic(int(topic_id))[:10])

    representative_docs = topic_model.get_representative_docs(int(topic_id))
    if representative_docs:
        print("Contoh dokumen:", representative_docs[0][:200] + "..."
              if len(representative_docs[0]) > 200 else representative_docs[0])

#Top2Vec

In [ ]:
!pip install top2vec gensim Sastrawi sentence-transformers

In [ ]:
from top2vec import Top2Vec
from sentence_transformers import SentenceTransformer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import pandas as pd

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

In [ ]:
import random
import numpy as np
import torch

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

load data

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/skripsi/bibit_reviews_labelled.csv")
documents = df["stopword"].astype(str).tolist()

model

In [ ]:
embedding_model = SentenceTransformer('indobenchmark/indobert-base-p2')

def indobert_embedding(documents):
    return embedding_model.encode(documents, show_progress_bar=True)

In [ ]:
model = Top2Vec(
    documents=documents,
    embedding_model=indobert_embedding,
    min_count=25,
    speed="learn",
    workers=1,
    umap_args={'n_neighbors': 15, 'n_components': 5, 'metric': 'cosine'},
    hdbscan_args={'min_cluster_size': 15, 'metric': 'euclidean'}
)

jumlah topik

In [ ]:
num_topics = model.get_num_topics()
print(f"Jumlah Topik Awal: {num_topics}")

kata representatif topik

In [ ]:
topic_words, word_scores, topic_nums = model.get_topics()

print("\nKata Representatif Tiap Topik:")
for i, words in zip(topic_nums, topic_words):
    print(f"Topik {i}: {words[:10]}")

In [ ]:
topic_words_list, word_scores_list, topic_nums_list = model.get_topics()

print("\nKata Representatif Sebelum Reduksi:")

for i, topic_id in enumerate(topic_nums_list):
    # Ensure we don't access out of bounds for words and scores
    if i < len(topic_words_list) and i < len(word_scores_list):
        words = topic_words_list[i]
        scores = word_scores_list[i]

        print(f"\nTopik {topic_id}")

        # Ambil top 10 kata langsung (sudah terurut dari Top2Vec)
        for j in range(min(10, len(words))):
            print(f"{words[j]} ({scores[j]:.4f})")
    else:
        print(f"\nTopik {topic_id}: No words or scores available.")

ukuran topik

In [ ]:
topic_sizes, topic_nums = model.get_topic_sizes()

print("\nUkuran Tiap Topik:")
for size, num in zip(topic_sizes, topic_nums):
    print(f"Topik {num}: {size} dokumen")

koherensi

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

# tokenisasi
tokenized_docs = [doc.split() for doc in documents]

# dictionary
dictionary = Dictionary(tokenized_docs)

coherence_per_topic = []
calculated_topic_ids = []

# Iterate through the topic numbers and their corresponding words
for i, topic_num in enumerate(topic_nums):
    words = topic_words[i]  # Get the words for the current topic

    # Ensure there are enough words for coherence calculation
    if len(words) == 0: # Gensim CoherenceModel needs at least one word per topic
        print(f"Skipping topic {topic_num} due to insufficient words for coherence calculation.")
        continue

    cm = CoherenceModel(
        topics=[list(words)],  # Ensure words is a list of strings
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )

    score = cm.get_coherence()

    coherence_per_topic.append(score)
    calculated_topic_ids.append(topic_num)  # Use the actual topic number

print("\nCoherence per Topik (Sebelum Reduksi)")
for i, topic_id in enumerate(calculated_topic_ids):
    print(f"Topik {topic_id}: {coherence_per_topic[i]:.4f}")

if len(coherence_per_topic) > 0:
    avg_coherence = sum(coherence_per_topic) / len(coherence_per_topic)
    print("\nRata-rata Coherence:")
    print(f"{avg_coherence:.4f}")
else:
    print("\nTidak ada topik yang valid untuk perhitungan coherence.")

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# ambil topik awal
# model.get_topics() returns (topic_words, word_scores, topic_nums)
topic_words_all, word_scores_all, topic_nums_all = model.get_topics()

freq = {}

# Iterate through each topic and its words/scores
for topic_idx in range(len(topic_nums_all)):
    words_for_current_topic = topic_words_all[topic_idx]
    scores_for_current_topic = word_scores_all[topic_idx]

    for word, score in zip(words_for_current_topic, scores_for_current_topic):
        # Accumulate scores for words across all topics
        freq[word] = freq.get(word, 0) + score

wc = WordCloud(
    width=1000,
    height=500,
    background_color="white"
)

wc.generate_from_frequencies(freq)

plt.figure(figsize=(12,6))
plt.imshow(wc)
plt.axis("off")
plt.title("WordCloud Top2Vec")
plt.show()

In [ ]:
topic_sizes, topic_nums = model.get_topic_sizes()

topic_distribution_t2v = pd.DataFrame({
    'Topic': topic_nums,
    'Count': topic_sizes
})

topic_distribution_t2v

reduksi topik hierarkis

In [ ]:
target_topics = 10

reduced_topics=model.hierarchical_topic_reduction(num_topics=target_topics)
print(f"Jumlah Topik Setelah Reduksi: {len(reduced_topics)}")

In [ ]:
#representasi kata topik
topic_words, word_scores, topic_nums = model.get_topics()

print("\nKata Representatif Setelah Reduksi:")
for i, topic_group in enumerate(reduced_topics):
    word_score_dict = {}

    for topic_id in topic_group:
        words = topic_words[topic_id]
        scores = word_scores[topic_id]

        for w, s in zip(words, scores):
            # kalau kata sama muncul lagi → ambil skor terbesar
            if w in word_score_dict:
                word_score_dict[w] = max(word_score_dict[w], s)
            else:
                word_score_dict[w] = s

    # urutkan berdasarkan skor tertinggi
    sorted_words = sorted(word_score_dict.items(), key=lambda x: x[1], reverse=True)

    # ambil top 10
    top_words = sorted_words[:10]

    print(f"\nTopik {i}")
    for w, s in top_words:
        print(f"{w} ({s:.4f})")

In [ ]:
#ukuran topik
size_dict = dict(zip(topic_nums, topic_sizes))

print("\nUkuran Topik Setelah Reduksi:")
for i, topic_group in enumerate(reduced_topics):
    total_size = sum(size_dict[topic_id] for topic_id in topic_group)
    print(f"Topik {i}: {total_size} dokumen")

In [ ]:
#koherensi
coherence_per_topic = []

print("\nCoherence Tiap Topik Setelah Reduksi:")

for i, words in enumerate(reduced_topics):
    cm = CoherenceModel(
        topics=[words],
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )

    score = cm.get_coherence()
    coherence_per_topic.append(score)

    print(f"Topik {i}: {score:.4f}")

avg_coherence_after = sum(coherence_per_topic) / len(coherence_per_topic)

print("\nRata-rata Coherence Setelah Reduksi:")
print(f"{avg_coherence_after:.4f}")

visualisasi

In [ ]:
reduced_topic_words = []
reduced_word_scores = []

for i, topic_group in enumerate(reduced_topics):
    word_score_dict = {}

    for topic_id in topic_group:
        words = topic_words[topic_id]
        scores = word_scores[topic_id]

        for w, s in zip(words, scores):
            # kalau kata sama muncul lagi → ambil skor terbesar
            if w in word_score_dict:
                word_score_dict[w] = max(word_score_dict[w], s)
            else:
                word_score_dict[w] = s

    # Urutkan dan ambil semua kata/skor untuk word cloud
    sorted_words = sorted(word_score_dict.items(), key=lambda x: x[1], reverse=True)
    words_for_group = [w for w, s in sorted_words]
    scores_for_group = [s for w, s in sorted_words]

    reduced_topic_words.append(words_for_group)
    reduced_word_scores.append(scores_for_group)


freq = {
    w: sc
    for words, scores in zip(reduced_topic_words, reduced_word_scores)
    for w, sc in zip(words, scores)
}

wc = WordCloud(background_color="white", width=800, height=400)\
    .generate_from_frequencies(freq)

plt.figure(figsize=(10,5))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud Top2Vec (Setelah Reduksi)")
plt.show()